# Task 4 - Speech-to-Reasoning Pipeline (Whisper + Quantized LLM)
**Goal:** Transcribe speech with Whisper, then run reasoning/QA with an Unsloth dynamic 4-bit model in one Colab notebook.

In [1]:
# Runtime pre-check (required for Whisper + Unsloth quantized LLM)

import sys

import torch



print("Python:", sys.version.split()[0])

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))

else:

    raise RuntimeError(

        "Use Google Colab GPU runtime to run this notebook end-to-end."

    )


Python: 3.14.3
CUDA available: False


RuntimeError: Use Google Colab GPU runtime to run this notebook end-to-end.

In [ ]:
# Install dependencies
!pip -q install unsloth transformers accelerate bitsandbytes datasets soundfile librosa

In [ ]:
import torch
from transformers import pipeline
from unsloth import FastLanguageModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

In [ ]:
# Whisper ASR pipeline
asr = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-small",
    device=0 if DEVICE == "cuda" else -1,
    chunk_length_s=20,
    batch_size=8,
)

print("Whisper loaded successfully.")

In [ ]:
# Example audio (you can replace with your own file upload)
!wget -q -O sample.wav https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac

transcription = asr("sample.wav")
transcribed_text = transcription["text"].strip()
print("Transcription:")
print(transcribed_text)

In [ ]:
# Load quantized reasoning model (dynamic 4-bit)
model_name = "unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

print("Reasoning model loaded:", model_name)

In [ ]:
@torch.inference_mode()
def reason_from_transcript(transcript: str, user_task: str = "Answer the question logically and briefly."):
    prompt = f"""You are a reasoning assistant.
Given the transcript below, perform the task.

Task: {user_task}
Transcript: {transcript}

Reasoned Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_seq_length).to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        use_cache=True
    )

    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    return generated[len(prompt):].strip() if generated.startswith(prompt) else generated

reasoning_result = reason_from_transcript(
    transcribed_text,
    user_task="Summarize the spoken sentence and identify the key message."
)

print("\nReasoning Output:")
print(reasoning_result)

## Optional: Use your own audio file
In Colab, upload an audio file and then run:
`transcription = asr("your_audio.wav")`
Then pass `transcription['text']` to the reasoning function.

## Notes for submission
- Capture screenshots for: Whisper output, quantized model load, and final reasoning answer.
- Mention handling of batching (`batch_size`), chunking (`chunk_length_s`), and 4-bit quantization for memory efficiency.